# TT-43: Backtesting Framework Playground

Interactive notebook for running backtests and inspecting results.

1. Configure a backtest (symbol, interval, date range)
2. Run the full pipeline: InfluxDB → Redis replay → HullMACD engine → BacktestSignal → InfluxDB
3. Query persisted BacktestSignals from InfluxDB
4. Display signal summary table
5. Overlay signals on Hull+MACD chart
6. Compare backtest signals vs live signals

**Data availability** (1-minute candles for entry pricing):
- `SPX{=m}`: from 2025-01-24
- `NVDA{=m}`: from 2025-02-11

In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import asyncio
import logging
from datetime import date, datetime, timedelta, timezone

import pandas as pd
import polars as pl
import influxdb_client

from tastytrade.common.logging import setup_logging
from tastytrade.config import RedisConfigManager
from tastytrade.connections import InfluxCredentials
from tastytrade.providers.market import MarketDataProvider
from tastytrade.providers.subscriptions import RedisSubscription
from tastytrade.messaging.models.events import CandleEvent

from tastytrade.backtest.models import BacktestConfig, BacktestSignal
from tastytrade.backtest.cli import run_backtest_orchestrated

from tastytrade.analytics.engines import HullMacdEngine
from tastytrade.analytics.indicators.momentum import macd
from tastytrade.analytics.visualizations.plots import (
    plot_macd_with_hull,
    VerticalLine,
)

In [3]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

logging.getLogger().handlers.clear()
setup_logging(level=logging.INFO, console=True, file=False)

## Connect to Data Sources

In [4]:
config = RedisConfigManager()
config.initialize()

influx_creds = InfluxCredentials(config=config)
influxdb = influxdb_client.InfluxDBClient(
    url=influx_creds.url,
    token=influx_creds.token,
    org=influx_creds.org,
)

subscription = RedisSubscription(config=config)
await subscription.connect()

streamer = MarketDataProvider(subscription, influxdb)
print("Connected to Redis and InfluxDB")

2026-02-25 23:48:45 - WARNING:tastytrade.config.manager:203:No variables found in .env file: .env
2026-02-25 23:48:45 - INFO:tastytrade.providers.subscriptions:70:Listening to Redis at redis://localhost:6379/0


Connected to Redis and InfluxDB


## Configure Backtest

Set the symbol, interval, and date range. The pricing interval auto-selects
to the next-lower granularity (5m signal → 1m pricing) unless overridden.

In [5]:
bt_config = BacktestConfig(
    symbol="SPX",
    signal_interval="5m",
    # pricing_interval=None,  # auto-selects "m" (1-minute)
    start_date=date(2025, 2, 18),
    end_date=date(2025, 2, 21),
)

print(f"Backtest ID:       {bt_config.backtest_id}")
print(f"Symbol:            {bt_config.symbol}")
print(f"Signal symbol:     {bt_config.signal_symbol}")
print(f"Pricing symbol:    {bt_config.pricing_symbol}")
print(f"Signal interval:   {bt_config.signal_interval}")
print(f"Pricing interval:  {bt_config.resolved_pricing_interval}")
print(f"Date range:        {bt_config.start_date} to {bt_config.end_date}")
print(f"Engine:            {bt_config.engine_type}")

Backtest ID:       29accf7e-00e1-4471-bc42-1831ef74ff5b
Symbol:            SPX
Signal symbol:     SPX{=5m}
Pricing symbol:    SPX{=m}
Signal interval:   5m
Pricing interval:  m
Date range:        2025-02-18 to 2025-02-21
Engine:            hull_macd


## Run Backtest

Executes the full pipeline:
1. **Replay** — downloads candles from InfluxDB, publishes to Redis
2. **Engine** — subscribes to Redis, runs HullMACD, publishes BacktestSignals
3. **Persist** — subscribes to BacktestSignal channel, writes to InfluxDB

All three run as concurrent async tasks within `run_backtest_orchestrated()`.

In [6]:
await run_backtest_orchestrated(bt_config)
print(f"\nBacktest complete — ID: {bt_config.backtest_id}")

2026-02-25 23:48:45 - INFO:tastytrade.backtest.cli:78:Prior close seeded: SPX{=5m} = 6114.63
2026-02-25 23:48:45 - INFO:tastytrade.backtest.cli:104:Starting orchestrated backtest — backtest_id=29accf7e-00e1-4471-bc42-1831ef74ff5b, symbol=SPX, signal=5m, pricing=m, range=2025-02-18 to 2025-02-21
2026-02-25 23:48:45 - INFO:tastytrade.providers.subscriptions:70:Listening to Redis at redis://localhost:6379/0
2026-02-25 23:48:45 - INFO:tastytrade.providers.subscriptions:92:Subscribed to backtest:CandleEvent:SPX{=5m} (type=CandleEvent)
2026-02-25 23:48:45 - INFO:tastytrade.backtest.runner:64:BacktestRunner subscribed to signal channel: backtest:CandleEvent:SPX{=5m}
2026-02-25 23:48:45 - INFO:tastytrade.providers.subscriptions:92:Subscribed to backtest:CandleEvent:SPX{=m} (type=CandleEvent)
2026-02-25 23:48:45 - INFO:tastytrade.backtest.runner:74:BacktestRunner subscribed to pricing channel: backtest:CandleEvent:SPX{=m}
2026-02-25 23:48:45 - INFO:tastytrade.backtest.runner:79:BacktestRunner r


Backtest complete — ID: 29accf7e-00e1-4471-bc42-1831ef74ff5b


## Query BacktestSignals from InfluxDB

Retrieve persisted BacktestSignal records using `measurement="BacktestSignal"`.

**InfluxDB Data Explorer query** (paste into `localhost:8086`):
```flux
from(bucket: "tastytrade")
  |> range(start: -30d)
  |> filter(fn: (r) => r._measurement == "BacktestSignal")
  |> filter(fn: (r) => r.eventSymbol == "SPX{=5m}")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> group()
  |> sort(columns: ["_time"])
```

In [7]:
backtest_signals = streamer.download_signals(
    symbol=bt_config.signal_symbol,
    start=bt_config.start_date,
    stop=bt_config.end_date + timedelta(days=1),
    measurement="BacktestSignal",
)
print(f"Retrieved {len(backtest_signals)} BacktestSignals from InfluxDB")

Retrieved 42 BacktestSignals from InfluxDB


## Signal Summary Table

In [8]:
if backtest_signals:
    signal_data = [
        {
            "time": s.start_time,
            "type": s.signal_type,
            "direction": s.direction,
            "trigger": s.trigger,
            "close_price": s.close_price,
            "entry_price": getattr(s, "entry_price", None),
            "hull_dir": s.hull_direction,
            "hull_value": round(s.hull_value, 2),
            "macd_hist": round(s.macd_histogram, 4),
        }
        for s in backtest_signals
    ]
    df_signals = pd.DataFrame(signal_data)
    display(df_signals)

    open_signals = [s for s in backtest_signals if s.signal_type == "OPEN"]
    close_signals = [s for s in backtest_signals if s.signal_type == "CLOSE"]
    bullish = [s for s in open_signals if s.direction == "BULLISH"]
    bearish = [s for s in open_signals if s.direction == "BEARISH"]
    print(f"\nOPEN: {len(open_signals)} (bullish={len(bullish)}, bearish={len(bearish)})")
    print(f"CLOSE: {len(close_signals)}")
else:
    print("No BacktestSignals found for this period.")

,time,type,direction,trigger,close_price,entry_price,hull_dir,hull_value,macd_hist
0,2025-02-18 14:30:00,OPEN,BULLISH,confluence,6122.64,None,Up,6115.49,0.5112
1,2025-02-18 15:00:00,OPEN,BEARISH,confluence,6113.90,None,Down,6119.34,-0.0071
2,2025-02-18 15:00:00,CLOSE,BULLISH,hull,6113.90,None,Down,6119.34,-0.0071
3,2025-02-18 15:40:00,CLOSE,BEARISH,hull,6117.15,None,Up,6112.84,-0.1878
4,2025-02-18 15:50:00,OPEN,BULLISH,confluence,6120.80,None,Up,6113.99,0.2450
5,2025-02-18 16:25:00,CLOSE,BULLISH,hull,6119.84,None,Down,6120.93,0.1588
6,2025-02-18 16:40:00,OPEN,BEARISH,confluence,6117.69,None,Down,6119.71,-0.0720
7,2025-02-18 16:45:00,CLOSE,BEARISH,macd,6122.25,None,Down,6119.59,0.1549
8,2025-02-18 16:50:00,OPEN,BULLISH,confluence,6122.83,None,Up,6119.89,0.3183
9,2025-02-18 17:00:00,OPEN,BEARISH,confluence,6117.57,None,Down,6119.77,-0.0923



OPEN: 21 (bullish=12, bearish=9)
CLOSE: 21


## Chart — Signals on Hull+MACD

Downloads the 5m candle data for the backtest period and overlays
BacktestSignal markers on the Hull+MACD chart.

In [9]:
# Download 5m candles for charting
candles_5m: pl.DataFrame = streamer.download(
    symbol=bt_config.signal_symbol,
    start=bt_config.start_date,
    stop=bt_config.end_date + timedelta(days=1),
    debug_mode=True,
)
candles_5m = candles_5m.filter(
    (pl.col("open") != 0)
    & (pl.col("high") != 0)
    & (pl.col("low") != 0)
    & (pl.col("close") != 0)
)
print(f"Loaded {candles_5m.height} candles for charting")

# Get prior close for indicator seeding
prior_day = streamer.get_daily_candle(
    bt_config.symbol,
    bt_config.start_date - timedelta(days=1),
)
print(f"Prior day close: {prior_day.close}")

# Compute MACD DataFrame
df_macd_5m = macd(
    candles_5m,
    prior_close=prior_day.close,
    fast_length=12,
    slow_length=26,
    macd_length=9,
)

# Build signal overlays
signal_lines = [
    VerticalLine(
        start_time=s.start_time,
        color="#55A868" if s.direction == "BULLISH" else "#C44E52",
        line_dash="dot" if s.signal_type == "CLOSE" else "solid",
        line_width=1.0 if s.signal_type == "OPEN" else 0.5,
        label=f"{s.signal_type} {s.direction}",
        opacity=0.6,
    )
    for s in backtest_signals
]

plot_macd_with_hull(
    df_macd_5m,
    pad_value=prior_day.close,
    start_time=datetime.combine(bt_config.start_date, datetime.min.time()),
    end_time=datetime.combine(bt_config.end_date + timedelta(days=1), datetime.min.time()),
    vertical_lines=signal_lines,
)

Loaded 1152 candles for charting
Prior day close: 6114.63


## Compare: Backtest vs Live Signals

Query both `BacktestSignal` and `TradeSignal` measurements for the same
symbol and period to compare backtest results against live production signals.

In [10]:
live_signals = streamer.download_signals(
    symbol=bt_config.signal_symbol,
    start=bt_config.start_date,
    stop=bt_config.end_date + timedelta(days=1),
    measurement="TradeSignal",
)
print(f"Live TradeSignals:    {len(live_signals)}")
print(f"Backtest signals:     {len(backtest_signals)}")

if live_signals and backtest_signals:
    print("\n--- Live Signals ---")
    for s in live_signals:
        print(
            f"  {s.signal_type:5s} {s.direction:7s} @ {s.start_time} "
            f"close={s.close_price:.2f} trigger={s.trigger}"
        )

    print("\n--- Backtest Signals ---")
    for s in backtest_signals:
        entry = getattr(s, "entry_price", None)
        entry_str = f"{entry:.2f}" if entry else "N/A"
        print(
            f"  {s.signal_type:5s} {s.direction:7s} @ {s.start_time} "
            f"close={s.close_price:.2f} entry={entry_str} trigger={s.trigger}"
        )

    # Overlay both on the same chart
    combined_lines = [
        VerticalLine(
            start_time=s.start_time,
            color="#1F77B4",  # blue for live
            line_dash="solid",
            line_width=1.5,
            label=f"LIVE {s.signal_type} {s.direction}",
            opacity=0.7,
        )
        for s in live_signals
    ] + [
        VerticalLine(
            start_time=s.start_time,
            color="#FF7F0E",  # orange for backtest
            line_dash="dash",
            line_width=1.0,
            label=f"BT {s.signal_type} {s.direction}",
            opacity=0.7,
        )
        for s in backtest_signals
    ]

    print("\nChart: Blue=Live, Orange=Backtest")
    plot_macd_with_hull(
        df_macd_5m,
        pad_value=prior_day.close,
        start_time=datetime.combine(bt_config.start_date, datetime.min.time()),
        end_time=datetime.combine(bt_config.end_date + timedelta(days=1), datetime.min.time()),
        vertical_lines=combined_lines,
    )
elif not live_signals:
    print("\nNo live TradeSignals found for this period — nothing to compare.")
else:
    print("\nNo backtest signals to compare.")

Live TradeSignals:    0
Backtest signals:     42

No live TradeSignals found for this period — nothing to compare.
